# Silver Transformation - factsalesorderline

This notebook builds the `factsalesorderline` dataset in the Silver layer.



## Run Shared Libraries



In [ ]:
%run ../Misc/sharedlibraries

In [ ]:
UpdatedDateTime = datetime.datetime.now()
Entity = "factsalesorderline"

## Read Source Tables



In [ ]:
salesorderlinedf = spark.table("bronze.salesorderline")
display(salesorderlinedf)

## Validate Source Data



In [ ]:
# display(spark.table("bronze.promotable"))

## Create Temporary View



In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW vwPromotable
AS
SELECT 
  PromotionId,
  CASE PromotionName
    WHEN 'Volume Discount 11 to 20' THEN 11
    WHEN 'Volume Discount 21 to 40' THEN 21
    WHEN 'Volume Discount 41 to 60' THEN 41
    WHEN 'Volume Discount > 60' THEN 61
    ELSE NULL
  END AS VolumeStart,
  CASE PromotionName
    WHEN 'Volume Discount 11 to 20' THEN 20
    WHEN 'Volume Discount 21 to 40' THEN 40
    WHEN 'Volume Discount 41 to 60' THEN 60
    WHEN 'Volume Discount > 60' THEN 9999999
    ELSE NULL
  END AS VolumeEnd,
  ValidFrom,
  ValidTo,
  PromoPercentage
FROM `bronze`.`promotable`
""")

display(spark.table("vwPromotable"))

## Create Temporary View



In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW vwFactSalesOrderLineBase
AS
SELECT 
  S.SalesOrderNumber,
  S.SalesOrderLine,
  CASE  
    WHEN isnull(S.LastProcessedChange_DateTime) THEN '1900-01-01'
    ELSE S.LastProcessedChange_DateTime 
  END AS LastProcessedChange_DateTime,

  from_utc_timestamp(S.DataLakeModified_DateTime,'CST') AS DataLakeModified_DateTime,

  S.ItemId,
  S.Qty,
  S.Price,
  S.Qty * S.Price AS TotalAmount,
  S.VatPercentage,

  C.CurrencyId,

  from_utc_timestamp(S.BookDate,'CST') AS BookDate,
  cast(date_format(S.BookDate,'yyyyMMdd') AS INT) AS BookDateKey,

  from_utc_timestamp(S.ShippedDate,'CST') AS ShippedDate,
  cast(date_format(S.ShippedDate,'yyyyMMdd') AS INT) AS ShippedDateKey,

  from_utc_timestamp(S.DeliveredDate,'CST') AS DeliveredDate,
  cast(date_format(S.DeliveredDate,'yyyyMMdd') AS INT) AS DeliveredKey,

  S.TrackingNumber,
  S.CustId,
  P.PaymentTypeId,
  PR.PromotionId,
  PR.PromoPercentage,
  S.RecordId

FROM `bronze`.`salesorderline` AS S

LEFT JOIN `bronze`.`currency` AS C 
  ON S.CurrencyCode = C.Code

LEFT JOIN `silver`.`dimpaymenttypes` AS P  
  ON S.PaymentTypeDesc = P.PaymentTypeDesc

LEFT JOIN vwPromotable AS PR 
  ON CASE 
       WHEN month(S.BookDate) = 1 
         THEN S.BookDate BETWEEN PR.ValidFrom AND PR.ValidTo
       ELSE S.Qty BETWEEN PR.VolumeStart AND PR.VolumeEnd  
     END
""")

display(spark.table("vwFactSalesOrderLineBase"))

## Apply Business Logic



In [ ]:
# COMMAND ----------

# MAGIC %md ###Apply Discount Calculation

# COMMAND ----------

spark.sql("""
CREATE OR REPLACE TEMP VIEW vwFactSalesOrderLineDiscount
AS
SELECT
  SalesOrderNumber,
  SalesOrderLine,
  LastProcessedChange_DateTime,
  DataLakeModified_DateTime,
  ItemId,
  Qty,
  Price,
  TotalAmount,

  CASE 
    WHEN PromotionId IS NULL THEN TotalAmount
    ELSE TotalAmount * (1 - PromoPercentage)
  END AS TotalAmountWithDiscount,

  VatPercentage,
  CurrencyId,
  BookDate,
  BookDateKey,
  ShippedDate,
  ShippedDateKey,
  DeliveredDate,
  DeliveredKey,
  TrackingNumber,
  CustId,
  PaymentTypeId,
  PromotionId,
  RecordId

FROM vwFactSalesOrderLineBase
""")

# display(spark.table("vwFactSalesOrderLineDiscount"))

## Create Temporary View



## Create Temporary View



In [ ]:
# %sql
# CREATE OR REPLACE TEMP VIEW vwFactSalesOrderLine
# AS
# SELECT 
#   SalesOrderNumber,
#   SalesOrderLine,
#   cast(LastProcessedChange_DateTime AS TIMESTAMP) AS LastProcessedChange_DateTime,
#   DataLakeModified_DateTime,
#   ItemId,
#   Qty,
#   Price,
#   TotalAmount,
#   TotalAmountWithDiscount,
#   VatPercentage,
#   TotalAmountWithDiscount * VatPercentage AS VatAmount,
#   TotalAmountWithDiscount + (TotalAmountWithDiscount * VatPercentage) AS TotalOrderAmount,
#   CurrencyId,
#   BookDate,
#   BookDateKey,
#   ShippedDate,
#   ShippedDateKey,
#   DeliveredDate,
#   DeliveredKey,
#   TrackingNumber,
#   CustId,
#   PaymentTypeId,
#   PromotionId,
#   current_timestamp() AS UpdatedDateTime,
#   xxhash64(RecordId) AS SalesOrderLineRecordId
# FROM vwFactSalesOrderLineDiscount

In [ ]:
display(spark.sql("SHOW VIEWS"))

In [ ]:
%sql
SHOW VIEWS;

In [ ]:
display(spark.sql("SELECT COUNT(*) AS row_count FROM vwPromotable"))
display(spark.sql("SELECT COUNT(*) AS row_count FROM vwFactSalesOrderLineBase"))

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW vwFactSalesOrderLineDiscount
AS
SELECT
  SalesOrderNumber,
  SalesOrderLine,
  LastProcessedChange_DateTime,
  DataLakeModified_DateTime,
  ItemId,
  Qty,
  Price,
  TotalAmount,

  CASE 
    WHEN PromotionId IS NULL THEN TotalAmount
    ELSE TotalAmount * (1 - PromoPercentage)
  END AS TotalAmountWithDiscount,

  VatPercentage,
  CurrencyId,
  BookDate,
  BookDateKey,
  ShippedDate,
  ShippedDateKey,
  DeliveredDate,
  DeliveredKey,
  TrackingNumber,
  CustId,
  PaymentTypeId,
  PromotionId,
  RecordId

FROM vwFactSalesOrderLineBase
""")

display(spark.table("vwFactSalesOrderLineDiscount"))

In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW vwFactSalesOrderLine
AS
SELECT 
  SalesOrderNumber,
  SalesOrderLine,

  cast(LastProcessedChange_DateTime AS TIMESTAMP) AS LastProcessedChange_DateTime,
  DataLakeModified_DateTime,

  ItemId,
  Qty,
  Price,
  TotalAmount,
  TotalAmountWithDiscount,

  VatPercentage,
  TotalAmountWithDiscount * VatPercentage AS VatAmount,
  TotalAmountWithDiscount + (TotalAmountWithDiscount * VatPercentage) AS TotalOrderAmount,

  CurrencyId,

  BookDate,
  BookDateKey,

  ShippedDate,
  ShippedDateKey,

  DeliveredDate,
  DeliveredKey,

  TrackingNumber,
  CustId,
  PaymentTypeId,
  PromotionId,

  current_timestamp() AS UpdatedDateTime,
  xxhash64(RecordId) AS SalesOrderLineRecordId

FROM vwFactSalesOrderLineDiscount
""")

factsalesorderlinedf = spark.table("vwFactSalesOrderLine")
display(factsalesorderlinedf)

In [ ]:
display(
    factsalesorderlinedf.selectExpr(
        "count(*) as total_rows",
        "count(CurrencyId) as rows_with_currency_id",
        "count(PaymentTypeId) as rows_with_payment_type_id",
        "count(PromotionId) as rows_with_promotion_id",
        "count(TotalAmount) as rows_with_total_amount",
        "count(TotalAmountWithDiscount) as rows_with_total_amount_with_discount",
        "count(VatAmount) as rows_with_vat_amount",
        "count(TotalOrderAmount) as rows_with_total_order_amount"
    )
)

In [ ]:
df_final = factsalesorderlinedf
saveDeltaTableToCatalog(df_final,"silver",Entity)

In [ ]:
display(spark.table("silver.factsalesorderline"))

In [ ]:
%sql
SELECT *
FROM silver.factsalesorderline
WHERE CurrencyId IS NULL
   OR PaymentTypeId IS NULL
   OR TotalOrderAmount IS NULL
LIMIT 20;

## Validate Before Write



## Prepare Final DataFrame



In [ ]:
df_final = factsalesorderlinedf

## Verify Source Schema



## Validate Silver Output



In [ ]:
display(
    spark.sql("""
        SELECT
          COUNT(*) AS total_rows,
          COUNT(CurrencyId) AS rows_with_currency_id,
          COUNT(PaymentTypeId) AS rows_with_payment_type_id,
          COUNT(PromotionId) AS rows_with_promotion_id,
          COUNT(TotalOrderAmount) AS rows_with_total_order_amount
        FROM silver.factsalesorderline
    """)
)

In [ ]:
%sql
SELECT *
FROM silver.factsalesorderline
WHERE CurrencyId IS NULL
   OR PaymentTypeId IS NULL
   OR TotalOrderAmount IS NULL
LIMIT 20;